### Import libraries

In [7]:
from pyspark.sql.functions import lit, to_date, col
from pyspark import SparkFiles
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, DecimalType
import requests
import base64
import os

StatementMeta(, 0e69e54c-5126-418c-a5cb-c73fde4f5850, 9, Finished, Available, Finished)

### Initialize parameters

In [4]:
GITHUB_PATH = (
    "https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales"
)

StatementMeta(, 0e69e54c-5126-418c-a5cb-c73fde4f5850, 6, Finished, Available, Finished)

### Load Addresses

In [14]:
file_name="Addresses.csv"

schema = StructType([
    StructField("address_id", IntegerType(), False),
    StructField("city", StringType(), True),
    StructField("postal_code", StringType(), True),
    StructField("street", StringType(), True),
    StructField("building", StringType(), True),
    StructField("country", StringType(), True),
    StructField("region", StringType(), True),
    StructField("address_type", StringType(), True),
    StructField("validity_start_date", StringType(), True),
    StructField("validity_end_date", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df_addresses = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df_addresses = df_addresses.withColumn("validity_start_date", to_date(col("validity_start_date"), "yyyyMMdd"))
df_addresses = df_addresses.withColumn("validity_end_date", to_date(col("validity_end_date"), "yyyyMMdd"))


StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 28, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/Addresses.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/Addresses.csv


### Load Employees and Employee Addresses

In [15]:
file_name="Employees.csv"

schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("first_name", StringType(), True),
    StructField("middle_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("initials", StringType(), True),
    StructField("country", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("language", StringType(), True),
    StructField("phone_no", StringType(), True),
    StructField("email", StringType(), True),
    StructField("login_name", StringType(), True),
    StructField("address_id", IntegerType(), True),
    StructField("validity_start_date", StringType(), True),
    StructField("validity_end_date", StringType(), True),
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("validity_start_date", to_date(col("validity_start_date"), "yyyyMMdd"))
df = df.withColumn("validity_end_date", to_date(col("validity_end_date"), "yyyyMMdd"))

df.write.format("delta").mode("overwrite").saveAsTable("employees")

df_employee_addresses = df_addresses.join(
    df,
    on="address_id",
    how="left_semi"
)

df_employee_addresses.write.format("delta").mode("overwrite").saveAsTable("employee_addresses")


StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 29, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/Employees.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/Employees.csv


### Load Customers and Customer Addresses

In [16]:
file_name="Customers.csv"

schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("email", StringType(), True),
    StructField("phone_no", StringType(), True),
    StructField("fax_no", StringType(), True),
    StructField("url", StringType(), True),
    StructField("address_id", IntegerType(), True),
    StructField("company_name", StringType(), True),
    StructField("legal_form", StringType(), True),
    StructField("created_by", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_by", IntegerType(), True),
    StructField("modified_date", StringType(), True),
    StructField("currency", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("created_date", to_date(col("created_date"), "yyyyMMdd"))
df = df.withColumn("modified_date", to_date(col("modified_date"), "yyyyMMdd"))

df.write.format("delta").mode("overwrite").saveAsTable("customers")

df_customer_addresses = df_addresses.join(
    df,
    on="address_id",
    how="left_semi"
)

df_customer_addresses.write.format("delta").mode("overwrite").saveAsTable("customer_addresses")


StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 30, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/Customers.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/Customers.csv


### Load Vendors and Vendor Addresses

In [17]:
file_name="Vendors.csv"

schema = StructType([
    StructField("vendor_id", IntegerType(), False),
    StructField("email", StringType(), True),
    StructField("phone_no", StringType(), True),
    StructField("fax_no", StringType(), True),
    StructField("url", StringType(), True),
    StructField("address_id", IntegerType(), True),
    StructField("company_name", StringType(), True),
    StructField("legal_form", StringType(), True),
    StructField("created_by", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_by", IntegerType(), True),
    StructField("modified_date", StringType(), True),
    StructField("currency", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("created_date", to_date(col("created_date"), "yyyyMMdd"))
df = df.withColumn("modified_date", to_date(col("modified_date"), "yyyyMMdd"))

df.write.format("delta").mode("overwrite").saveAsTable("vendors")

df_vendor_addresses = df_addresses.join(
    df,
    on="address_id",
    how="left_semi"
)

df_vendor_addresses.write.format("delta").mode("overwrite").saveAsTable("vendor_addresses")


StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 31, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/Vendors.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/Vendors.csv


### Load Product Categories

In [18]:
file_name="ProductCategories.csv"

schema = StructType([
    StructField("product_category_id", StringType(), False),
    StructField("created_by", IntegerType(), True),
    StructField("created_date", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("created_date", to_date(col("created_date"), "yyyyMMdd"))

df.write.format("delta").mode("overwrite").saveAsTable("product_categories")

StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 32, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/ProductCategories.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/ProductCategories.csv


### Load Product Category Texts

In [19]:
file_name="ProductCategoryTexts.csv"

schema = StructType([
    StructField("product_category_id", StringType(), False),
    StructField("language", StringType(), True),
    StructField("short_description", StringType(), True),
    StructField("medium_description", StringType(), True),
    StructField("long_description", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df.write.format("delta").mode("overwrite").saveAsTable("product_category_texts")

StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 33, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/ProductCategoryTexts.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/ProductCategoryTexts.csv


### Load Products

In [20]:
file_name="Products.csv"

schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("type_code", StringType(), True),
    StructField("product_category_id", StringType(), True),
    StructField("created_by", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_by", IntegerType(), True),
    StructField("modified_date", StringType(), True),
    StructField("vendor_id", IntegerType(), True),
    StructField("tax_tariff_code", StringType(), True),
    StructField("quantity_unit", StringType(), True),
    StructField("weight_measure", DoubleType(), True),
    StructField("weight_unit", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("price", DecimalType(), True),
    StructField("width", DoubleType(), True),
    StructField("depth", DoubleType(), True),
    StructField("height", DoubleType(), True),
    StructField("dimension_unit", StringType(), True),
    StructField("product_pic_url", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("created_date", to_date(col("created_date"), "yyyyMMdd"))
df = df.withColumn("modified_date", to_date(col("modified_date"), "yyyyMMdd"))

df.write.format("delta").mode("overwrite").saveAsTable("products")


StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 34, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/Products.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/Products.csv


### Load Product Texts

In [21]:
file_name="ProductTexts.csv"

schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("language", StringType(), False),
    StructField("short_description", StringType(), True),
    StructField("medium_description", StringType(), True),
    StructField("long_description", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df.write.format("delta").mode("overwrite").saveAsTable("product_texts")

StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 35, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/ProductTexts.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770206403212_0001/spark-31a17055-b50e-414c-9b18-a1acd9ae5967/userFiles-e0d3a63f-3240-4691-9bbc-58e0636f0fe6/ProductTexts.csv


### Load Sales Orders

In [8]:
file_name="SalesOrders.csv"

schema = StructType([
    StructField("sales_order_id", IntegerType(), False),
    StructField("created_by", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_by", IntegerType(), True),
    StructField("modified_date", StringType(), True),
    StructField("fisc_variant", StringType(), True),
    StructField("fiscal_year_period", StringType(), True),
    StructField("note_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), False),
    StructField("sales_org", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("gross_amount", DecimalType(), True),
    StructField("net_amount", DecimalType(), True),
    StructField("tax_amount", DecimalType(), True),
    StructField("lifecycle_status", StringType(), True),
    StructField("billing_status", StringType(), True),
    StructField("delivery_status", StringType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("created_date", to_date(col("created_date"), "yyyyMMdd"))
df = df.withColumn("modified_date", to_date(col("modified_date"), "yyyyMMdd"))

df = df.withColumn(
    "lifecycle_status",
    F.when(F.col("lifecycle_status") == "I", "In Process")
     .when(F.col("lifecycle_status") == "C", "Completed")
     .when(F.col("lifecycle_status") == "X", "Cancelled")
     .otherwise("Unknown")
)

df = df.withColumn(
    "billing_status",
    F.when(F.col("billing_status") == "N", "Not Billed")
     .when(F.col("billing_status") == "P", "Partially Billed")
     .when(F.col("billing_status") == "C", "Completely Billed")
     .when(F.col("billing_status") == "X", "Cancelled")
     .otherwise("Unknown")
)

df = df.withColumn(
    "delivery_status",
    F.when(F.col("delivery_status") == "N", "Not Delivered")
     .when(F.col("delivery_status") == "P", "Partially Delivered")
     .when(F.col("delivery_status") == "C", "Completely Delivered")
     .when(F.col("delivery_status") == "X", "Cancelled")
     .otherwise("Unknown")
)

df.write.format("delta").mode("overwrite").saveAsTable("sales_orders")


StatementMeta(, 0e69e54c-5126-418c-a5cb-c73fde4f5850, 10, Finished, Available, Finished)

'Downloading https://raw.githubusercontent.com/martinabrle/demo-data/main/Sample_Bike_Sales/SalesOrders.csv'

Downloaded data file:///mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1770216896704_0001/spark-b99ebdc2-42b6-4c16-893f-f9971ca8603f/userFiles-76e059f7-50fc-4638-90e1-44b3a4df8523/SalesOrders.csv


### Load Sales Order Items

In [9]:
file_name="SalesOrderItems.csv"

schema = StructType([
    StructField("sales_order_id", IntegerType(), False),
    StructField("sales_order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), False),
    StructField("note_id", IntegerType(), True),
    StructField("currency", StringType(), True),
    StructField("gross_amount", DecimalType(), True),
    StructField("net_amount", DecimalType(), True),
    StructField("tax_amount", DecimalType(), True),
    StructField("item_atp_status", StringType(), True),
    StructField("op_item_pos", StringType(), True),
    StructField("quantity", DecimalType(), True),
    StructField("quantity_unit", DecimalType(), True),
    StructField("delivery_date", DateType(), True)
])

gh_full_file_name = f"{GITHUB_PATH}/{file_name}"
display(f"Downloading {gh_full_file_name}")

sc.addFile(gh_full_file_name)
gh_file_name  = 'file://' +SparkFiles.get(file_name)

df = spark.read.csv(path=gh_file_name,header=True,schema=schema)
print(f"Downloaded data {gh_file_name}")

df = df.withColumn("delivery_date", to_date(col("delivery_date"), "yyyyMMdd"))

df = df.withColumn(
    "item_atp_status_text",
    F.when(F.col("item_atp_status") == "C", "Confirmed")
     .when(F.col("item_atp_status") == "P", "Partially Confirmed")
     .when(F.col("item_atp_status") == "N", "Not Confirmed")
     .when(F.col("item_atp_status") == "X", "Cancelled / Not Relevant")
     .otherwise("Unknown")

df.write.format("delta").mode("overwrite").saveAsTable("sales_order_items")


StatementMeta(, 0e69e54c-5126-418c-a5cb-c73fde4f5850, 11, Finished, Available, Finished)

SyntaxError: '(' was never closed (2165240333.py, line 30)

In [11]:
spark.sql("TRUNCATE TABLE customer_addresses")
spark.sql("TRUNCATE TABLE customers")
spark.sql("TRUNCATE TABLE employee_addresses")
spark.sql("TRUNCATE TABLE employees")
spark.sql("TRUNCATE TABLE product_categories")
spark.sql("TRUNCATE TABLE product_category_texts")
spark.sql("TRUNCATE TABLE product_texts")
spark.sql("TRUNCATE TABLE products")
spark.sql("TRUNCATE TABLE sales_order_items")
spark.sql("TRUNCATE TABLE sales_orders")
spark.sql("TRUNCATE TABLE vendor_addresses")
spark.sql("TRUNCATE TABLE vendors")

StatementMeta(, e6cf73bc-46af-4134-b40e-760563cc657c, 25, Finished, Available, Finished)

DataFrame[num_affected_rows: bigint]